# 47 · The QR codes in a worked example's write-up, checked against OpenCV

## What this checks

A worked example's write-up carries two QR codes. One opens Fieldwork with the example loaded. The other opens the
example's own files. This check covers the codes Fieldwork release 0.200.0 prints. It checks no widget: the codes
belong to the write-up.

A code that does not read back to its address sends a reader nowhere. So it is worth checking.

## How it is checked

**OpenCV**, a widely used library for reading images, reads each code Fieldwork draws. It has two detectors: the
classic one, and one built on ArUco markers. Its authors did not write Fieldwork's encoder, qrcode-generator. No code
is shared with Fieldwork.

A second encoder, **segno**, makes its own code for each address. It uses the same size and the same level of error
correction. Error correction is the spare data that lets a damaged code still be read.

We draw each of Fieldwork's codes with the blank margin the QR standard asks for: four squares wide.

The data are 28 addresses: every address Fieldwork would print for the catalog examples and the built-in ones.

OpenCV is not in the browser kernel, so this check runs in CPython only. In the browser, the notebook says so and
skips the rest.

## What should happen

1. **Every code reads back** to its address, exactly, by at least one of OpenCV's two detectors.
2. **segno agrees on the size.** It makes a code of the same version, so the same number of squares, for every
   address.
3. **segno's own codes read back too**, by at least one detector. This shows OpenCV can read a code that Fieldwork's
   encoder did not make. A failure on Fieldwork's codes alone would then be Fieldwork's.

We also report, without requiring it, whether segno draws the very same pattern. Both encoders choose a pattern by
the standard's own rule. A different pattern is still a valid code, as long as the first criterion holds.

These criteria were fixed before the check was run.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
try:
    from checks.check_47_qr_links import run
except ImportError as missing:
    run = None
    print("Not runnable here:", missing, "- OpenCV is not in the browser kernel; run in CPython.")

if run:
    report = run(Path.cwd() / "fixtures" / "qr-links-001.json")
    print("agrees:", report["agrees"])
    print(report["addresses"], "addresses;", report["samePatternAsSegno"], "drawn identically by segno")
    for code in report["codes"][:4]:
        print(" ", code["readBack"], code["text"])
    print("differs:", report["differs"])

## The result on record

First recorded on 10 November 2026, against Fieldwork release 0.79.0. Rerun on 26 November 2026, against Fieldwork
release 0.200.0: **Agrees**. Fieldwork's record of widget checks does not list this check, because the codes belong
to no widget.

All 28 codes read back to their addresses, by both detectors. segno made codes of the same size, and its own codes
read back too. One of segno's codes was read only by the ArUco-based detector. segno drew none of the 28 with the
very same pattern. That is allowed: each is still a valid code.

**Why two detectors.** Until 26 November 2026 the check used the classic detector alone. It could not read segno's
code for one address added since the first run, while it read Fieldwork's. That was a limit of the decoder, not of
Fieldwork, but the check counted it as a disagreement. Each code is now read by both detectors, as check 52 already
did.

Running the cell above repeats the check on the same data.

## What this does not show

- That a phone camera reads a printed code at a given size and light.
- That an address answers, or that the pinned commit is the one the write-up was made from. Those are checked when
  the write-up is exported, not here.
- Anything in the browser kernel. OpenCV is not there, so this ran in CPython only.

In [ ]:
if run:
    for item in report["notEstablished"]:
        print("-", item)
    e = report["environment"]
    print("opencv:", e["opencv"], "· segno:", e["segno"], "·", e["runtime"])